# Chunking Practical: Most Useful Methods

This notebook covers the following methods:

1. `RecursiveCharacterTextSplitter`
2. `TokenTextSplitter`
3. `CharacterTextSplitter`
4. Fixed-size chunking
5. Content-aware chunking
6. Document structure-based chunking
7. Semantic chunking

## Learning objectives

- Explain chunk size and chunk overlap
- Compare fixed, character, recursive, token, structural, and semantic splitting
- Understand why different strategies produce different boundaries
- Choose a suitable starting method for a given document type
- Evaluate chunks instead of blindly selecting one splitter

1. Fixed-size character chunking
   → CharacterTextSplitter

2. Fixed-size token chunking
   → TokenTextSplitter

3. Recursive structure-aware chunking
   → RecursiveCharacterTextSplitter

4. Document structure-based chunking
   → MarkdownHeaderTextSplitter, HTML splitters, etc.

5. Semantic chunking
   → Embedding-based topic boundaries

CharacterTextSplitter

→ Primarily separator-based

→ Length measured using characters

TokenTextSplitter

→ Directly splits according to tokens

RecursiveCharacterTextSplitter

→ Tries to preserve paragraphs, lines and words

→ Not purely blind fixed-size chunking

In [2]:
from __future__ import annotations
import math
import re
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

Matplotlib is building the font cache; this may take a moment.
/Users/rakhi/Documents/AI/AgenticAI/Practical/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Sample Document

The sample document intentionally contains multiple topics and Markdown headings.  
This makes it useful for comparing basic, structure-aware, and semantic chunking.

In [13]:
document = """
# Artificial Intelligence

Artificial intelligence enables machines to perform tasks that normally require human intelligence.
AI systems can analyze information, identify patterns, and support decision-making.

## Machine Learning

Machine learning is a branch of artificial intelligence.
It enables systems to learn patterns from historical data instead of relying only on explicitly programmed rules.
Supervised learning uses labelled examples, while unsupervised learning discovers patterns in unlabelled data.

## Deep Learning

Deep learning uses neural networks containing multiple layers.
It is commonly applied to image recognition, speech processing, and natural-language applications.

# Employee Leave Policy

Employees receive twenty paid leave days during a calendar year.
Leave requests should be submitted to the reporting manager before the planned absence.
Emergency leave may be submitted later when advance notice is not possible.

## Approval Process

The manager reviews the leave balance, project schedule, and team availability.
Approved leave is recorded in the human-resource management system.

# Information Security

Employees must not share passwords or authentication codes.
Sensitive company information should be stored only in approved systems.
Suspected security incidents must be reported to the security team immediately.
""".strip()

In [14]:
from pypdf import PdfReader
reader = PdfReader("data/llama2-research-paper.pdf")
page_texts = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        page_texts.append(text)

    print(
        f"Page {page_number}: "
        f"{len(text)} characters extracted"
    )

Page 1: 1892 characters extracted
Page 2: 2452 characters extracted
Page 3: 3687 characters extracted
Page 4: 2508 characters extracted
Page 5: 2778 characters extracted
Page 6: 2108 characters extracted
Page 7: 3792 characters extracted
Page 8: 3145 characters extracted
Page 9: 4015 characters extracted
Page 10: 4970 characters extracted
Page 11: 4137 characters extracted
Page 12: 3720 characters extracted
Page 13: 3598 characters extracted
Page 14: 2563 characters extracted
Page 15: 4218 characters extracted
Page 16: 3179 characters extracted
Page 17: 4583 characters extracted
Page 18: 3304 characters extracted
Page 19: 2677 characters extracted
Page 20: 4867 characters extracted
Page 21: 2931 characters extracted
Page 22: 3630 characters extracted
Page 23: 3540 characters extracted
Page 24: 4881 characters extracted


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Page 25: 2363 characters extracted
Page 26: 3332 characters extracted
Page 27: 4053 characters extracted
Page 28: 3226 characters extracted
Page 29: 4443 characters extracted
Page 30: 2468 characters extracted
Page 31: 2521 characters extracted
Page 32: 3617 characters extracted
Page 33: 2002 characters extracted
Page 34: 2772 characters extracted
Page 35: 4655 characters extracted
Page 36: 5048 characters extracted
Page 37: 4796 characters extracted
Page 38: 5116 characters extracted
Page 39: 4640 characters extracted
Page 40: 4421 characters extracted
Page 41: 4153 characters extracted
Page 42: 4675 characters extracted
Page 43: 4334 characters extracted
Page 44: 4365 characters extracted
Page 45: 159 characters extracted
Page 46: 3469 characters extracted
Page 47: 3821 characters extracted
Page 48: 2543 characters extracted
Page 49: 1892 characters extracted
Page 50: 2220 characters extracted
Page 51: 3130 characters extracted
Page 52: 2583 characters extracted
Page 53: 1473 charact

In [15]:
document = "\n\n".join(page_texts)

In [16]:
print(document)

Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic
Sergey Edunov

# 2. Helper Functions

These helpers print chunks consistently and collect simple statistics.

In [18]:
from typing import Any

def print_chunks(
    title: str,
    chunks: list[Any],
    preview_limit: int = 500,
) -> None:
    print(f"\n{'=' * 90}")
    print(title)
    print(f"Total chunks: {len(chunks)}")
    print("=" * 90)

    for index, chunk in enumerate(chunks, start=1):

        # Get text from a LangChain Document or normal Python object
        text = getattr(
            chunk,
            "page_content",
            str(chunk),
        )

        # Get metadata if available
        metadata = getattr(
            chunk,
            "metadata",
            {},
        )

        print(
            f"\nChunk {index} | "
            f"characters={len(text)}"
        )

        if metadata:
            print("Metadata:", metadata)

        if len(text) > preview_limit:
            print(text[:preview_limit] + "...")
        else:
            print(text)

In [19]:
from typing import Any
import numpy as np


def chunk_stats(
    method: str,
    chunks: list[Any],
) -> dict[str, Any]:

    lengths = [
        len(
            getattr(
                chunk,
                "page_content",
                str(chunk),
            )
        )
        for chunk in chunks
    ]

    return {
        "method": method,
        "number_of_chunks": len(chunks),
        "minimum_length": min(lengths) if lengths else 0,
        "maximum_length": max(lengths) if lengths else 0,
        "average_length": (
            round(float(np.mean(lengths)), 2)
            if lengths
            else 0
        ),
        "total_stored_characters": sum(lengths),
    }

In [20]:
## Variable declartion to store the results of different chunking methods
all_results: dict[str, list[Any]] = {}

# 3. Fixed-Size Chunking

Fixed-size chunking divides content after a specified number of characters or tokens.

It is simple and predictable, but it may break:

- Words
- Sentences
- Paragraphs
- Logical topics

In [21]:
## Own logic to perfomre fixed size chunking 

def fixed_size_chunking(text: str, chunk_size: int, chunk_overlap: int = 0) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must satisfy 0 <= chunk_overlap < chunk_size."
        )

    step = chunk_size - chunk_overlap

    return [
        text[start:start + chunk_size]
        for start in range(0, len(text), step)
    ]

In [22]:
text = "ABCDEFGHIJKLMNOPQRSTUVWXYZ" 

fixed_size_chunking(
    text=text,
    chunk_size=10,
    chunk_overlap=2,
)

['ABCDEFGHIJ', 'IJKLMNOPQR', 'QRSTUVWXYZ', 'YZ']

In [23]:
document

'Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev\nPunit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich\nYinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra\nIgor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi\nAlan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang\nRoss Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang\nAngela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic\

In [24]:
fixed_chunks = fixed_size_chunking(
    text=document,
    chunk_size=300,
    chunk_overlap=50,
)

In [25]:
fixed_chunks

['Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernand',
 'ya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev\nPunit Singh Koura Marie-Anne Lachaux Thibaut Lavr',
 '\nPunit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich\nYinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra\nIgor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi\nAlan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqi',
 'Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tan

In [26]:
len(fixed_chunks)

1043

In [27]:
all_results["Manual fixed-size"] = fixed_chunks

In [28]:
print_chunks("Manual Fixed-Size Chunking", fixed_chunks)


Manual Fixed-Size Chunking
Total chunks: 1043

Chunk 1 | characters=300
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernand

Chunk 2 | characters=300
ya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavr

Chunk 3 | characters=300

Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smit

## What to observe

- Every chunk targets the same maximum character size.
- Fifty characters are repeated between neighbouring chunks.
- A boundary may occur in the middle of a sentence or heading.
- Overlap protects some boundary information but also creates duplication.

# 4. `CharacterTextSplitter`


`CharacterTextSplitter` splits using one selected separator.

In this example, the separator is a blank line (`"\n\n"`), so it tries to keep paragraphs together.

This is easy to understand, but one separator may not be sufficient for every document. This time we use `RecursiveCharacterTextSplitter` to define multiple seperators

In [42]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,TokenTextSplitter, MarkdownHeaderTextSplitter
)

In [ ]:
# By default, seperator is set to "\n\n" and chunk_size is set to 1000 characters.
character_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
)

In [31]:
character_chunks = character_splitter.split_text(document)

Created a chunk of size 1892, which is longer than the specified 300
Created a chunk of size 2452, which is longer than the specified 300
Created a chunk of size 3687, which is longer than the specified 300
Created a chunk of size 2508, which is longer than the specified 300
Created a chunk of size 2778, which is longer than the specified 300
Created a chunk of size 2108, which is longer than the specified 300
Created a chunk of size 3792, which is longer than the specified 300
Created a chunk of size 3145, which is longer than the specified 300
Created a chunk of size 4015, which is longer than the specified 300
Created a chunk of size 4970, which is longer than the specified 300
Created a chunk of size 4137, which is longer than the specified 300
Created a chunk of size 3720, which is longer than the specified 300
Created a chunk of size 3598, which is longer than the specified 300
Created a chunk of size 2563, which is longer than the specified 300
Created a chunk of size 4218, whic

Above if we you see it saying like "Created a chunk of size 2369, which is longer than the specified 300". It giving prefernce to seperator not chunk_size

In [32]:
all_results["CharacterTextSplitter"] = character_chunks

In [33]:
print_chunks("CharacterTextSplitter", character_chunks)


CharacterTextSplitter
Total chunks: 77

Chunk 1 | characters=1892
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev...

Chunk 2 | characters=2452
Contents
1 Introduction 3
2 Pretraining 5
2.1 Pretraining Data . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5
2.2 Training Details . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5
2.3 Llama 2 Pretrained Model Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 7
3 Fine-tuning 8
3.1 Supervis

**Important observation:**

`CharacterTextSplitter` is separator-driven. If an individual separator-delimited block is already larger than the target size, the output may not behave like strict fixed-width slicing.

Use it when the source has a predictable separator and you want a simple baseline.

# 5. `RecursiveCharacterTextSplitter`

This splitter tries separators recursively.

Typical order:

```text
Paragraph → line → sentence-like boundary → word → character
```

It first tries to preserve large natural units. If a unit is too large, it moves to a finer separator.

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    add_start_index=True,
)

## First it uses the \n\n separator to split the text. If the resulting chunks are still too large, 
# it will then use the \n separator, and so on, until it reaches the last separator (an empty string), 
# which will split the text into individual characters if necessary.

In [36]:
text = """Artificial Intelligence

AI enables machines to perform intelligent tasks.
Machine learning is a part of AI.

Employee Leave Policy

Employees receive 20 paid leave days.
Leave must be approved by the manager.
"""

recursive_splitter.create_documents(
    texts=[text],
    metadatas=[{"source": "classroom_sample.md"}],
)

[Document(metadata={'source': 'classroom_sample.md', 'start_index': 0}, page_content='Artificial Intelligence\n\nAI enables machines to perform intelligent tasks.\nMachine learning is a part of AI.\n\nEmployee Leave Policy\n\nEmployees receive 20 paid leave days.\nLeave must be approved by the manager.')]

In [37]:
recursive_documents = recursive_splitter.create_documents(
    texts=[document],
    metadatas=[{"source": "classroom_sample.md"}],
)

In [38]:
all_results["RecursiveCharacterTextSplitter"] = recursive_documents

In [39]:
print_chunks(
    "RecursiveCharacterTextSplitter",
    recursive_documents,
)


RecursiveCharacterTextSplitter
Total chunks: 1102

Chunk 1 | characters=257
Metadata: {'source': 'classroom_sample.md', 'start_index': 0}
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen

Chunk 2 | characters=242
Metadata: {'source': 'classroom_sample.md', 'start_index': 258}
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev

Chunk 3 | characters=229
Metadata: {'source': 'classroom_sample.md', 'start_index': 501}
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poul

**Why it is commonly used**

- It works well for general text
- It tries to preserve natural boundaries
- It still controls chunk size
- It is a strong baseline for PDFs, reports, policies, and articles

# 6. `TokenTextSplitter`

LLMs and embedding models process tokens rather than raw characters.

Token-based chunking is useful when:

- The embedding model has a strict token limit
- You need predictable model-input sizing
- Character counts do not accurately represent model usage

This example uses the `cl100k_base` tokenizer.

In [43]:
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=80,
    chunk_overlap=15,
)

In [44]:
token_chunks = token_splitter.split_text(document)
all_results["TokenTextSplitter"] = token_chunks

In [45]:
print_chunks("TokenTextSplitter", token_chunks)


TokenTextSplitter
Total chunks: 1113

Chunk 1 | characters=260
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Gu

Chunk 2 | characters=257
ikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian

Chunk 3 | characters=247
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin

Chunk 4 | characters=273
aylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta 

**Character count is not token count**

A 300-character chunk is not necessarily a 300-token chunk.

Token count varies with:

- Language
- Punctuation
- Numbers
- Source code
- Model tokenizer

# 7. `Content-Aware Chunking`

Content-aware chunking respects natural content boundaries such as:

- Paragraphs
- Sentences
- Sections
- Lists

The custom implementation below:

1. Splits the document into paragraphs
2. Keeps a paragraph intact when possible
3. Uses sentence-level splitting when a paragraph is too large
4. Merges units until the target size is reached

This is an educational implementation, not a replacement for every production parser.

In [51]:
def split_sentences(text: str) -> list[str]:
    return [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", text.strip())
        if sentence.strip()
    ]

In [57]:
import re
def content_aware_chunking(
    text: str,
    max_characters: int = 300,
) -> list[str]:
    if max_characters <= 0:
        raise ValueError("max_characters must be greater than zero.")

    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\\n\\n")
        if paragraph.strip()
    ]

    logical_units: list[str] = []

    for paragraph in paragraphs:
        if len(paragraph) <= max_characters:
            logical_units.append(paragraph)
        else:
            logical_units.extend(split_sentences(paragraph))

    chunks: list[str] = []
    current_units: list[str] = []
    current_length = 0

    for unit in logical_units:
        separator_length = 2 if current_units else 0
        proposed_length = current_length + separator_length + len(unit)

        if current_units and proposed_length > max_characters:
            chunks.append("\\n\\n".join(current_units))
            current_units = [unit]
            current_length = len(unit)
        else:
            current_units.append(unit)
            current_length = proposed_length

    if current_units:
        chunks.append("\\n\\n".join(current_units))

    return chunks

In [58]:
text = """Artificial Intelligence

AI enables machines to perform intelligent tasks.
Machine learning is a part of AI.

Employee Leave Policy

Employees receive 20 paid leave days.
Leave must be approved by the manager.
"""

In [59]:
content_aware_chunking(text = text)

['Artificial Intelligence\n\nAI enables machines to perform intelligent tasks.\nMachine learning is a part of AI.\n\nEmployee Leave Policy\n\nEmployees receive 20 paid leave days.\nLeave must be approved by the manager.']

In [60]:
content_aware_chunks = content_aware_chunking(
    text=document,
    max_characters=300,
)

In [61]:
print_chunks("Content-Aware Chunking", content_aware_chunks)


Content-Aware Chunking
Total chunks: 1042

Chunk 1 | characters=1212
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev...

Chunk 2 | characters=276
Our fine-tuned LLMs, calledLlama 2-Chat, are optimized for dialogue use cases.\n\nOur
models outperform open-source chat models on most benchmarks we tested, and based on
our human evaluations for helpfulness and safety, may be a suitable substitute for closed-
source models.

Chunk 3 | characters=212
We provide a detailed description of our approach to fine-tuning and safety
improvements ofLlama 

In [62]:
all_results["Content-aware"] = content_aware_chunks

**Difference from blind fixed-size splitting**

Fixed-size chunking asks:

> Has the size limit been reached?

Content-aware chunking also asks:

> Is this a natural place to create a boundary?

# 8. Document Structure-Based Chunking

Some documents already contain meaningful structure:

- Markdown headings
- HTML heading tags
- JSON hierarchy
- Code functions and classes
- PDF layout elements

For Markdown, `MarkdownHeaderTextSplitter` can use headings to create logical sections and preserve heading context as metadata.

In [63]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ],
    strip_headers=True,
)

header_documents = markdown_splitter.split_text(document)

all_results["Markdown structure"] = header_documents
print_chunks(
    "MarkdownHeaderTextSplitter",
    header_documents,
)


MarkdownHeaderTextSplitter
Total chunks: 1

Chunk 1 | characters=260607
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev...


**Two-stage production pattern**

A heading-based section may still be too large.

A practical strategy is:

```text
Stage 1: Split by document structure
Stage 2: Recursively split oversized sections
```

In [64]:
secondary_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)

structure_then_recursive = secondary_splitter.split_documents(
    header_documents
)

all_results["Structure + recursive"] = structure_then_recursive
print_chunks(
    "Structure-Based Splitting Followed by Recursive Splitting",
    structure_then_recursive,
)


Structure-Based Splitting Followed by Recursive Splitting
Total chunks: 1277

Chunk 1 | characters=167
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra

Chunk 2 | characters=249
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou

Chunk 3 | characters=228
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra

Chunk 4 | characters=175
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh 